# PLD_accounting Tutorial: Privacy Accounting for Random Allocation

This notebook demonstrates how to use the `PLD_accounting` package to compute tight (ε, δ)-differential privacy guarantees for the random allocation subsampling scheme using Privacy Loss Distributions (PLDs).

## References

- **Repository**: [https://github.com/moshenfeld/PLD_accounting](https://github.com/moshenfeld/PLD_accounting)
- **PyPI**: [https://pypi.org/project/PLD-accounting/](https://pypi.org/project/PLD-accounting/)
- **Paper**: [https://arxiv.org/pdf/2602.17284](https://arxiv.org/pdf/2602.17284)

## Installation

Run the cell below to install:

In [ ]:
# Uncomment and run this cell to install the package from PyPI
!pip install PLD_accounting

# Or install from source:
# !git clone https://github.com/moshenfeld/PLD_accounting.git
# !cd PLD_accounting && pip install .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.5/131.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: attrs
    Found existing installation: attrs 25.4.0
    Uninstalling attrs-25.4.0:
      Successfully uninstalled attrs-25.4.0


In [ ]:
# Import required libraries
import numpy as np

from PLD_accounting.types import (
    PrivacyParams,
    AllocationSchemeConfig,
    Direction,
    BoundType,
)
from PLD_accounting import (
    gaussian_allocation_PLD,
    gaussian_allocation_epsilon_extended,
    gaussian_allocation_epsilon_range,
    gaussian_allocation_delta_extended,
    gaussian_allocation_delta_range,
    subsample_PLD,
)

In [ ]:
from PLD_accounting.random_allocation_api import numerical_allocation_epsilon_range
delta = 1e-6
sigma = 3.0
num_steps = 100
# Epsilon query: Find epsilon for a given delta
# epsilon_accuracy=-1 means: use 10% of the Poisson-estimated epsilon as target accuracy
epsilon_upper, epsilon_lower = numerical_allocation_epsilon_range(
    sigma=sigma,
    num_steps=num_steps,
    delta=delta,
    epsilon_accuracy=0.01
)


In [ ]:
delta = 1e-6
sigma = 3.0
num_steps = 100
num_selected = 10
# Epsilon query: Find epsilon for a given delta
# epsilon_accuracy=-1 means: use 10% of the Poisson-estimated epsilon as target accuracy
epsilon_upper, epsilon_lower = numerical_allocation_epsilon_range(
    sigma=sigma,
    num_steps=num_steps,
    delta=delta,
    num_selected=num_selected,
    epsilon_accuracy=0.01
)


## Example 1: Simple API for $t$-out-of-$k$ allocation

**Parameters**:

- **sigma ($\sigma$)**: Gaussian noise scale
- **num_steps ($t$)**: Total number of computation steps
- **num_selected ($k$)**: Number of allocations
- **delta ($\delta$)**: Target delta for $(\varepsilon, \delta)$-DP (for epsilon queries)
- **epsilon ($\varepsilon$)**: Target epsilon for $(\varepsilon, \delta)$-DP (for delta queries)

**Optional parameters**:
- **epsilon_accuracy**: Target width of the output $\varepsilon$ range
- **delta_accuracy**: Target width of the output $\delta$ range

**Output**:
- **Epsilon queries**: `numerical_allocation_epsilon_range()` returns `(epsilon_upper, epsilon_lower)` for given $\delta$
- **Delta queries**: `numerical_allocation_delta_range()` returns `(delta_upper, delta_lower)` for given $\varepsilon$

In [ ]:
# Example 1: Adaptive epsilon and delta computation with automatic resolution refinement
delta = 1e-6
sigma = 3.0
num_steps = 100
num_selected = 10

# Epsilon query: Find epsilon for a given delta
# epsilon_accuracy=-1 means: use 10% of the Poisson-estimated epsilon as target accuracy
epsilon_upper, epsilon_lower = numerical_allocation_epsilon_range(
    sigma=sigma,
    num_steps=num_steps,
    delta=delta,
    num_selected=num_selected,
    epsilon_accuracy=-1,  # negative value → 10% of Poisson guess
)

print("[Example 1] Adaptive range computation with automatic resolution tuning:\n")
print(f"  Parameters: σ={sigma}, num_steps={num_steps}, num_selected={num_selected}")
print(f"\nEpsilon query (for δ={delta:.0e}):")
print(f"  Epsilon upper bound: {epsilon_upper:.4f}")
print(f"  Epsilon lower bound: {epsilon_lower:.4f}")
print(f"  Gap: {epsilon_upper - epsilon_lower:.4f} ({100*(epsilon_upper - epsilon_lower)/epsilon_upper:.2f}%)")
print(f"  ✓ Privacy guarantee: (ε ∈ [{epsilon_lower:.4f}, {epsilon_upper:.4f}], δ={delta:.0e})")

# Delta query: Find delta for a given epsilon
target_epsilon = 2.0
delta_upper, delta_lower = numerical_allocation_delta_range(
    sigma=sigma,
    num_steps=num_steps,
    epsilon=target_epsilon,
    num_selected=num_selected,
    delta_accuracy=-1,  # negative value → 10% of Poisson guess
)

print(f"\nDelta query (for ε={target_epsilon}):")
print(f"  Delta upper bound: {delta_upper:.2e}")
print(f"  Delta lower bound: {delta_lower:.2e}")
print(f"  ✓ Privacy guarantee: (ε={target_epsilon}, δ ∈ [{delta_lower:.2e}, {delta_upper:.2e}])")

## Example 2: Struct-based API for $t$-out-of-$k$ allocation

This example uses the struct-based API for full control over resolution parameters.


### `PrivacyParams` — Training Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `sigma` | float | Gaussian noise scale |
| `num_steps` | int | Total number of computation steps |
| `num_selected` | int | Number of allocations |
| `num_epochs` | int | Training epochs (usually 1) |
| `delta` | float | Target delta for $(\varepsilon, \delta)$-DP (for epsilon queries) |
| `epsilon` | float | Target epsilon for $(\varepsilon, \delta)$-DP (for delta queries) |

**Note**: Set either `delta` (for epsilon queries) or `epsilon` (for delta queries), not both.

### `AllocationSchemeConfig` — Resolution & Accuracy

| Parameter | Type | Description |
|-----------|------|-------------|
| `loss_discretization` | float | Grid spacing for PLD (smaller = tighter bounds, slower, more memory) |
| `tail_truncation` | float | Probability mass to truncate from distribution tails (smaller = more accurate) |
| `max_grid_FFT` | int | Maximum grid size for FFT-based convolution |
| `max_grid_mult` | int | Maximum grid size for multiplication-based convolution |

### `Direction` — Which Privacy Loss Direction to Analyze

- **`Direction.REMOVE`**: Privacy loss when removing a record from the dataset
- **`Direction.ADD`**: Privacy loss when adding a record to the dataset
- **`Direction.BOTH`**: Analyze both directions (most common)

### `BoundType` — Upper vs Lower Bounds

- **`BoundType.DOMINATES`**: Upper bound on privacy loss (pessimistic, **safe for privacy proofs**)
- **`BoundType.IS_DOMINATED`**: Lower bound on privacy loss (optimistic, for research)

In [ ]:
# Example 2: Set up privacy parameters
delta = 1e-6
sigma = 3.0
num_steps = 100
num_selected = 10

# Epsilon query: Compute epsilon for a given delta
params_for_epsilon = PrivacyParams(
    sigma=sigma,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=1,
    delta=delta,
)

config = AllocationSchemeConfig(
    loss_discretization=0.05,     # Coarser = faster but looser bounds
    tail_truncation=delta * 0.1,  # Truncate small probability mass
    max_grid_mult=50_000,         # Maximum grid size for convolution
)

epsilon = numerical_allocation_epsilon(
    params=params_for_epsilon,
    config=config,
    direction=Direction.BOTH,      # Analyze both ADD and REMOVE
    bound_type=BoundType.DOMINATES, # Upper bound (safe for proofs)
)

print("[Example 2] Direct computation with full control:\n")
print(f"  Parameters: σ={sigma}, num_steps={num_steps}, num_selected={num_selected}")
print(f"  Configuration: loss_discretization={config.loss_discretization}, tail_truncation={config.tail_truncation:.0e}")
print(f"\nEpsilon query (for δ={delta:.0e}):")
print(f"  Epsilon: {epsilon:.4f}")

# Delta query: Compute delta for a given epsilon
target_epsilon = 2.0
params_for_delta = PrivacyParams(
    sigma=sigma,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=1,
    epsilon=target_epsilon,
)

delta_result = numerical_allocation_delta(
    params=params_for_delta,
    config=config,
    direction=Direction.BOTH,
    bound_type=BoundType.DOMINATES,
)

print(f"\nDelta query (for ε={target_epsilon}):")
print(f"  Delta: {delta_result:.2e}")

print(f"\n  ✓ Privacy guarantees: (ε={epsilon:.4f}, δ={delta:.0e}) and (ε={target_epsilon}, δ={delta_result:.2e})")

[Example 2] Direct computation with full control:

  Parameters: σ=3.0, num_steps=100, num_selected=10
  Configuration: loss_discretization=0.05, tail_truncation=1e-07

Epsilon query (for δ=1e-06):
  Epsilon: 1.5485

Delta query (for ε=2.0):
  Delta: 9.81e-08

  ✓ Privacy guarantees: (ε=1.5485, δ=1e-06) and (ε=2.0, δ=9.81e-08)


## Example 3: Reusable PLD object for exploring $(\varepsilon, \delta)$ tradeoffs

This example shows how to compute a Privacy Loss Distribution (PLD) once and query it multiple times.

**Output**:
- Returns a `PLD` object that can be queried multiple times:
  - `pld.get_epsilon_for_delta(delta)` → Find $\varepsilon$ for a given $\delta$
  - `pld.get_delta_for_epsilon(epsilon)` → Find $\delta$ for a given $\varepsilon$

In [ ]:
# Create a PLD object (Privacy Loss Distribution)
pld = allocation_PLD(
    params=params_for_epsilon,
    config=config,
    direction=Direction.BOTH,
    bound_type=BoundType.DOMINATES,
)

print("[Example 3] Reusable PLD for multiple queries:\n")

# Epsilon queries: Epsilon for multiple delta values
print("Epsilon queries (δ → ε):")
for target_delta in [1e-4, 1e-5, 1e-6]:
    eps = pld.get_epsilon_for_delta(target_delta)
    print(f"  δ={target_delta:.0e}  →  ε={eps:.4f}")

# Delta queries: Delta for multiple epsilon values
print("\nDelta queries (ε → δ):")
for target_epsilon in [1.5, 2.0, 2.5]:
    delta_val = pld.get_delta_for_epsilon(target_epsilon)
    print(f"  ε={target_epsilon}  →  δ={delta_val:.2e}")

[Example 3] Reusable PLD for multiple queries:

Epsilon queries (δ → ε):
  δ=1e-04  →  ε=1.1540
  δ=1e-05  →  ε=1.3614
  δ=1e-06  →  ε=1.5485

Delta queries (ε → δ):
  ε=1.5  →  δ=1.84e-06
  ε=2.0  →  δ=9.81e-08
  ε=2.5  →  δ=9.81e-08


## Example 4: PREAMBLE-style subsampling + composition

This example models federated learning with subsampling and composition across multiple rounds.

**Additional parameters**:
- **q**: Subsampling probability (each client participates with 5% chance)
- **num_rounds**: Total training rounds = 50

**Privacy accounting pipeline**:
1. **Base PLD** → Compute privacy for one training round
2. **Subsample** → Amplify privacy via subsampling with probability $q$
3. **Compose** → Accumulate privacy loss across $R$ rounds  
4. **Query** → Get final $\varepsilon$ for target $\delta$


In [ ]:
# Training setup
sigma        = 3.0    # Large sigma → compact PLD, faster computation
num_steps    = 100    # Steps inside one round
num_selected = 10     # Clients selected per step
q            = 0.05   # Subsampling probability per round
num_rounds   = 50     # Total number of rounds composed

# Scale config to account for composition and subsampling
# loss_discretization scales as 1/sqrt(num_rounds * q) (central-limit-theorem-like)
# tail_truncation must cover the full composed tail, so it scales as 1/(num_rounds * q)
scale = num_rounds * q
config_preamble = AllocationSchemeConfig(
    loss_discretization=config.loss_discretization / np.sqrt(scale),
    tail_truncation=config.tail_truncation / scale,
    max_grid_mult=config.max_grid_mult,
)

preamble_params = PrivacyParams(
    sigma=sigma,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=1,
    delta=delta,
)

print("[Example 4] PREAMBLE-style: subsample + compose workflow")
print(f"  Scenario: σ={sigma}, t={num_steps}, k={num_selected}, q={q}, R={num_rounds}, δ={delta:.0e}\n")

# Step 1: Get PLD for one training round
print(f"  [1/4] Computing base PLD for one round...")
base_pld = allocation_PLD(
    params=preamble_params,
    config=config_preamble,
    direction=Direction.BOTH,
)

# Step 2: Apply subsampling amplification
print(f"  [2/4] Applying subsampling amplification (q={q})...")
subsampled = subsample_PLD(
    pld=base_pld,
    sampling_probability=q,
    bound_type=BoundType.DOMINATES,
)

# Step 3: Compose across multiple rounds
print(f"  [3/4] Composing across {num_rounds} rounds...")
composed = subsampled.self_compose(num_rounds)

# Step 4: Get final epsilon
print(f"  [4/4] Querying final epsilon...")
eps_preamble = composed.get_epsilon_for_delta(delta)

print(f"\n  ✓ Final privacy guarantee: (ε={eps_preamble:.4f}, δ={delta:.0e})")
print(f"    Full parameters: σ={sigma}, t={num_steps}, k={num_selected}, q={q}, R={num_rounds}")

[Example 4] PREAMBLE-style: subsample + compose workflow
  Scenario: σ=3.0, t=100, k=10, q=0.05, R=50, δ=1e-06

  [1/4] Computing base PLD for one round...
  [2/4] Applying subsampling amplification (q=0.05)...
  [3/4] Composing across 50 rounds...
  [4/4] Querying final epsilon...

  ✓ Final privacy guarantee: (ε=0.6362, δ=1e-06)
    Full parameters: σ=3.0, t=100, k=10, q=0.05, R=50
